In [1]:
from transformers import AutoTokenizer, EsmForMaskedLM


# noinspection PyNoneFunctionAssignment
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
model = EsmForMaskedLM.from_pretrained("facebook/esm2_t33_650M_UR50D")

model.eval()

/Users/dcanevarollo/.conda/envs/variant-prediction/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 539/539 [00:00<00:00, 24949.29it/s]


EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 1280, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (rotary_embeddings): EsmRotaryEmbedding()
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-32): 33 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=1280, out_features=1280, bias=True)
              (key): Linear(in_features=1280, out_features=1280, bias=True)
              (value): Linear(in_features=1280, out_features=1280, bias=True)
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_features=1280, ou

In [2]:
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device.type}")

model.to(device)

Using device: cpu


EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 1280, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (rotary_embeddings): EsmRotaryEmbedding()
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-32): 33 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=1280, out_features=1280, bias=True)
              (key): Linear(in_features=1280, out_features=1280, bias=True)
              (value): Linear(in_features=1280, out_features=1280, bias=True)
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_features=1280, ou

In [3]:
seq = "MKTIIALSYIFCLVFADYKDDDDA"

# noinspection PyCallingNonCallable
inputs = tokenizer(seq, return_tensors="pt")
inputs = { k: v.to(device) for k, v in inputs.items() }

print(inputs)

{'input_ids': tensor([[ 0, 20, 15, 11, 12, 12,  5,  4,  8, 19, 12, 18, 23,  4,  7, 18,  5, 13,
         19, 15, 13, 13, 13, 13,  5,  2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}


In [4]:
with torch.no_grad():
    outputs = model(**inputs)

outputs.logits.shape

torch.Size([1, 26, 33])

In [5]:
probs = torch.softmax(outputs.logits, dim=-1)
probs.shape

torch.Size([1, 26, 33])

In [6]:
wt_id = tokenizer.convert_tokens_to_ids("A")
mut_id = tokenizer.convert_tokens_to_ids("V")

In [7]:
prob_wt = probs[0, 6, wt_id]
prob_mut = probs[0, 6, mut_id]

In [8]:
log_ratio = float(torch.log(prob_mut) - torch.log(prob_wt))
print(log_ratio)

-2.275240898132324
